In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.metrics import r2_score, mean_squared_error

np.random.seed(42)


# ============================================================
# 1) PARAMETERS
# ============================================================

def get_params():
    media_channels = ["tv", "search", "social", "video"]
    non_media_channels = ["promo_index", "distribution_index", "seasonality_index"]

    adstock_alpha = {"tv": 0.75, "search": 0.45, "social": 0.35, "video": 0.55}
    hill_k = {"tv": 2.8, "search": 2.2, "social": 2.0, "video": 2.4}
    hill_s = {"tv": 2.4, "search": 2.2, "social": 2.6, "video": 2.3}

    # Priors
    roi_priors_media = {"tv": 1.8, "search": 3.0, "social": 2.4, "video": 2.0}
    contribution_priors_non_media = {"promo_index": 0.45, "distribution_index": 0.35, "seasonality_index": 0.20}

    return {
        "n_periods": 156,
        "freq": "W",
        "media_channels": media_channels,
        "non_media_channels": non_media_channels,
        "adstock_alpha": adstock_alpha,
        "hill_k": hill_k,
        "hill_s": hill_s,
        "roi_priors_media": roi_priors_media,
        "contribution_priors_non_media": contribution_priors_non_media,
        "nonmedia_expected_share_of_units": 0.35,   # for prior mapping
        "l2_strength": 60.0,                        # stronger => closer to priors
        "noise_sd_units": 1500
    }


# ============================================================
# 2) CORE TRANSFORMS
# ============================================================

def geometric_adstock(x, alpha):
    out = np.zeros_like(x, dtype=float)
    out[0] = x[0]
    for t in range(1, len(x)):
        out[t] = x[t] + alpha * out[t - 1]
    return out


def hill_transform(x, k, s):
    x_safe = np.clip(x, 1e-9, None)
    num = np.power(x_safe, s)
    den = num + np.power(k, s)
    return num / den


def median_scale(x):
    med = np.median(x)
    return (x / med if med != 0 else x.copy()), med


def median_inverse(x_scaled, med):
    return x_scaled * med


def zscore_scale(x):
    mu = np.mean(x)
    sd = np.std(x)
    sd = sd if sd > 0 else 1.0
    return (x - mu) / sd, mu, sd


def zscore_inverse(z, mu, sd):
    return z * sd + mu


# ============================================================
# 3) DATA GENERATION
# ============================================================

def generate_original_data(n_periods=156, freq="W", noise_sd_units=1500):
    idx = pd.date_range("2023-01-01", periods=n_periods, freq=freq)
    n = len(idx)
    t = np.arange(n)

    # Media impressions
    tv_imp = 15_000_000 + 2_000_000*np.sin(2*np.pi*t/52 + 0.3) + np.random.normal(0, 1_100_000, n)
    se_imp =  9_000_000 + 1_200_000*np.sin(2*np.pi*t/26 + 0.8) + np.random.normal(0,   700_000, n)
    so_imp =  7_000_000 + 1_100_000*np.sin(2*np.pi*t/13 + 1.2) + np.random.normal(0,   650_000, n)
    vi_imp =  8_000_000 + 1_300_000*np.sin(2*np.pi*t/39 + 0.5) + np.random.normal(0,   750_000, n)

    tv_imp = np.clip(tv_imp, 1_000_000, None)
    se_imp = np.clip(se_imp,   600_000, None)
    so_imp = np.clip(so_imp,   500_000, None)
    vi_imp = np.clip(vi_imp,   700_000, None)

    # Spend from CPM
    tv_sp = (tv_imp / 1000.0) * np.random.normal(11.5, 0.7, n)
    se_sp = (se_imp / 1000.0) * np.random.normal(9.0, 0.5, n)
    so_sp = (so_imp / 1000.0) * np.random.normal(7.5, 0.4, n)
    vi_sp = (vi_imp / 1000.0) * np.random.normal(10.0, 0.6, n)

    # Non-media
    promo = np.clip(np.random.normal(0.5, 0.15, n), 0.05, 1.0)
    dist  = np.clip(0.75 + 0.05*np.sin(2*np.pi*t/52 + 2.2) + np.random.normal(0, 0.03, n), 0.55, 0.95)
    seas  = 0.5 + 0.5*np.sin(2*np.pi*t/52 - 1.1)

    # Price
    price = 9.5 + 0.35*np.sin(2*np.pi*t/52 + 0.2) + np.random.normal(0, 0.1, n)
    price = np.clip(price, 8.8, 10.5)

    # Synthetic true units/revenue (for training target)
    base_units = 120_000
    units = (
        base_units
        + 1800*np.log1p(tv_imp/1e6)
        + 2200*np.log1p(se_imp/1e6)
        + 1400*np.log1p(so_imp/1e6)
        + 1600*np.log1p(vi_imp/1e6)
        + 1500*((promo - promo.mean())/promo.std())
        + 1000*((dist  - dist.mean())/dist.std())
        + 1200*((seas  - seas.mean())/seas.std())
        + np.random.normal(0, noise_sd_units, n)
    )
    units = np.clip(units, 1000, None)
    revenue = units * price

    df = pd.DataFrame({
        "date": idx,
        "tv_impressions": tv_imp, "tv_spend": tv_sp,
        "search_impressions": se_imp, "search_spend": se_sp,
        "social_impressions": so_imp, "social_spend": so_sp,
        "video_impressions": vi_imp, "video_spend": vi_sp,
        "promo_index": promo,
        "distribution_index": dist,
        "seasonality_index": seas,
        "price": price,
        "revenue": revenue
    })
    return df


# ============================================================
# 4) FEATURE ENGINEERING
# ============================================================

def build_features(df, p):
    out = df.copy()
    media_medians = {}
    non_media_stats = {}

    # inverse revenue->units (requested step)
    out["units"] = out["revenue"] / out["price"]

    # media: median scaling -> adstock -> hill
    for c in p["media_channels"]:
        imp_col = f"{c}_impressions"
        scaled, med = median_scale(out[imp_col].values)
        media_medians[c] = med

        ads = geometric_adstock(scaled, p["adstock_alpha"][c])
        h = hill_transform(ads, p["hill_k"][c], p["hill_s"][c])

        out[f"{c}_scaled"] = scaled
        out[f"{c}_adstock"] = ads
        out[f"{c}_hill"] = h

        # inverse scaled (for traceability)
        out[f"{c}_scaled_inverse"] = median_inverse(scaled, med)

    # non-media: zscore only (no adstock/hill)
    for c in p["non_media_channels"]:
        z, mu, sd = zscore_scale(out[c].values)
        non_media_stats[c] = (mu, sd)
        out[f"{c}_scaled"] = z
        out[f"{c}_scaled_inverse"] = zscore_inverse(z, mu, sd)

    return out, media_medians, non_media_stats


# ============================================================
# 5) PRIORS (ROI/CONTRIB -> COEF PRIORS)
# ============================================================

def build_prior_means_stds(df, p):
    media = p["media_channels"]
    non_media = p["non_media_channels"]

    avg_price = df["price"].mean()

    # Media prior means from ROI priors
    mu_media, sd_media = [], []
    for c in media:
        roi = p["roi_priors_media"][c]                 # revenue/spend
        units_per_spend = roi / avg_price              # units/spend
        avg_spend = df[f"{c}_spend"].mean()
        avg_hill  = df[f"{c}_hill"].mean()

        mu = units_per_spend * (avg_spend / max(avg_hill, 1e-9))
        sd = 0.5                    # tighter -> higher precision
        mu_media.append(mu)
        sd_media.append(sd)

    # Non-media prior means from contribution shares
    shares = p["contribution_priors_non_media"].copy()
    ssum = sum(shares.values())
    shares = {k: v/ssum for k, v in shares.items()}

    expected_nonmedia_units = p["nonmedia_expected_share_of_units"] * df["units"].mean()

    mu_non, sd_non = [], []
    for c in non_media:
        target_units = shares[c] * expected_nonmedia_units
        avg_abs_x = np.mean(np.abs(df[f"{c}_scaled"].values)) + 1e-9
        mu = target_units / avg_abs_x
        sd = 0.5
        mu_non.append(mu)
        sd_non.append(sd)

    return np.array(mu_media), np.array(sd_media), np.array(mu_non), np.array(sd_non)


# ============================================================
# 6) PRIOR-CENTERED RIDGE
# ============================================================

def ridge_prior_objective(beta, X, y, mu_prior, sigma_prior, l2_strength):
    resid = y - X @ beta
    data_loss = np.sum(resid**2)

    precision = 1.0 / np.clip(sigma_prior, 1e-9, None)**2
    prior_penalty = np.sum(precision * (beta - mu_prior)**2)

    return data_loss + l2_strength * prior_penalty


def fit_prior_centered_ridge(df, p):
    media = p["media_channels"]
    non_media = p["non_media_channels"]

    y = df["units"].values

    X_media = np.column_stack([df[f"{c}_hill"].values for c in media])
    X_non = np.column_stack([df[f"{c}_scaled"].values for c in non_media])
    X_core = np.column_stack([X_media, X_non])

    # intercept column
    X = np.column_stack([np.ones(len(df)), X_core])

    mu_m, sd_m, mu_n, sd_n = build_prior_means_stds(df, p)

    mu_intercept = np.array([y.mean()])
    sd_intercept = np.array([10.0 * y.std() + 1e-6])  # weak regularization on intercept

    mu_prior = np.concatenate([mu_intercept, mu_m, mu_n])
    sd_prior = np.concatenate([sd_intercept, sd_m, sd_n])

    x0 = mu_prior.copy()

    res = minimize(
        ridge_prior_objective,
        x0=x0,
        args=(X, y, mu_prior, sd_prior, p["l2_strength"]),
        method="L-BFGS-B"
    )
    if not res.success:
        raise RuntimeError(f"Optimization failed: {res.message}")

    beta = res.x
    y_hat = X @ beta

    metrics = {
        "r2_units": r2_score(y, y_hat),
        "rmse_units": np.sqrt(mean_squared_error(y, y_hat))
    }

    intercept = beta[0]
    b_media = beta[1:1+len(media)]
    b_non = beta[1+len(media):]

    return intercept, b_media, b_non, y_hat, metrics, mu_prior, sd_prior


# ============================================================
# 7) CONTRIBUTIONS + OUTPUT TABLES
# ============================================================

def build_outputs(df, p, intercept, b_media, b_non, y_hat):
    out = df.copy()
    media = p["media_channels"]
    non_media = p["non_media_channels"]

    # channel contributions in units
    for i, c in enumerate(media):
        out[f"contrib_{c}_units"] = b_media[i] * out[f"{c}_hill"].values

    for i, c in enumerate(non_media):
        out[f"contrib_{c}_units"] = b_non[i] * out[f"{c}_scaled"].values

    # baseline and totals
    out["baseline_units"] = intercept
    contrib_unit_cols = [f"contrib_{c}_units" for c in media + non_media]
    out["incremental_units"] = out[contrib_unit_cols].sum(axis=1)
    out["predicted_units"] = out["baseline_units"] + out["incremental_units"]

    # convert to revenue using price
    for c in media + non_media:
        out[f"contrib_{c}_revenue"] = out[f"contrib_{c}_units"] * out["price"]

    out["baseline_revenue"] = out["baseline_units"] * out["price"]
    out["incremental_revenue"] = out[[f"contrib_{c}_revenue" for c in media + non_media]].sum(axis=1)
    out["predicted_revenue"] = out["baseline_revenue"] + out["incremental_revenue"]

    # weekly contribution file
    weekly_cols = ["date", "price", "revenue", "predicted_revenue", "baseline_revenue", "incremental_revenue"]
    for c in media + non_media:
        weekly_cols += [f"contrib_{c}_units", f"contrib_{c}_revenue"]
    weekly_df = out[weekly_cols].copy()

    # ROI file
    roi_rows = []
    for c in media:
        inc_rev = out[f"contrib_{c}_revenue"].sum()
        spend = out[f"{c}_spend"].sum()
        roi = inc_rev / spend if spend > 0 else np.nan
        roi_rows.append({
            "channel": c,
            "type": "media",
            "total_spend": spend,
            "total_incremental_revenue": inc_rev,
            "roi": roi
        })

    for c in non_media:
        inc_rev = out[f"contrib_{c}_revenue"].sum()
        roi_rows.append({
            "channel": c,
            "type": "non_media",
            "total_spend": np.nan,
            "total_incremental_revenue": inc_rev,
            "roi": np.nan
        })

    roi_df = pd.DataFrame(roi_rows).sort_values(["type", "channel"]).reset_index(drop=True)

    return out, weekly_df, roi_df


# ============================================================
# 8) RESPONSE CURVES (one sheet per media channel)
# ============================================================

def response_curve_media_channel(df, p, channel, beta_media, n_points=60):
    imp_col = f"{channel}_impressions"
    spend_col = f"{channel}_spend"

    p5, p95 = np.percentile(df[imp_col].values, [5, 95])
    imp_grid = np.linspace(max(1.0, 0.5*p5), 1.5*p95, n_points)

    med = np.median(df[imp_col].values)
    x_sc = imp_grid / med if med != 0 else imp_grid
    x_ads = x_sc / (1 - p["adstock_alpha"][channel])   # steady-state approx
    x_hill = hill_transform(x_ads, p["hill_k"][channel], p["hill_s"][channel])

    incr_units = beta_media * x_hill
    incr_revenue = incr_units * df["price"].mean()

    avg_cpm = np.mean((df[spend_col].values / np.clip(df[imp_col].values, 1e-9, None)) * 1000.0)
    spend_grid = (imp_grid / 1000.0) * avg_cpm

    roi = incr_revenue / np.clip(spend_grid, 1e-9, None)

    # Marginal ROI: derivative dRevenue/dSpend (finite-difference)
    d_rev = np.diff(incr_revenue)
    d_spend = np.diff(spend_grid)
    mroi_mid = d_rev / np.clip(d_spend, 1e-9, None)

    # same length as grid: pad ends
    mroi = np.empty_like(spend_grid)
    mroi[0] = mroi_mid[0]
    mroi[-1] = mroi_mid[-1]
    if len(mroi) > 2:
        mroi[1:-1] = 0.5 * (mroi_mid[:-1] + mroi_mid[1:])

    return pd.DataFrame({
        "impressions": imp_grid,
        "spend": spend_grid,
        "predicted_incremental_units": incr_units,
        "predicted_incremental_revenue": incr_revenue,
        "implied_roi": roi,
        "mroi": mroi
    })


def write_response_curves_excel(df, p, b_media, out_file="./outputs/response_curves.xlsx"):
    with pd.ExcelWriter(out_file, engine="xlsxwriter") as writer:
        for i, c in enumerate(p["media_channels"]):
            curve = response_curve_media_channel(df, p, c, b_media[i], n_points=60)
            curve.to_excel(writer, sheet_name=c[:31], index=False)


# ============================================================
# 9) MAIN
# ============================================================

def run_pipeline():
    p = get_params()

    # original data
    original = generate_original_data(
        n_periods=p["n_periods"],
        freq=p["freq"],
        noise_sd_units=p["noise_sd_units"]
    )
    original.to_csv("./outputs/original_data.csv", index=False)

    # features
    feat, media_medians, non_media_stats = build_features(original, p)

    # model
    intercept, b_media, b_non, y_hat, metrics, mu_prior, sd_prior = fit_prior_centered_ridge(feat, p)

    # outputs
    full_df, weekly_df, roi_df = build_outputs(feat, p, intercept, b_media, b_non, y_hat)
    weekly_df.to_csv("./outputs/weekly_contribution.csv", index=False)
    roi_df.to_csv("./outputs/roi.csv", index=False)
    write_response_curves_excel(full_df, p, b_media, out_file="./outputs/response_curves.xlsx")

    print("Model metrics:", metrics)
    print("Saved files:")
    print("- original_data.csv")
    print("- weekly_contribution.csv")
    print("- roi.csv")
    print("- response_curves.xlsx")


if __name__ == "__main__":
    run_pipeline()

Model metrics: {'r2_units': -69.74648639598227, 'rmse_units': np.float64(20238.703920878776)}
Saved files:
- original_data.csv
- weekly_contribution.csv
- roi.csv
- response_curves.xlsx


In [2]:
p = get_params()

In [ ]:
original = generate_original_data(
    n_periods=p["n_periods"],
    freq=p["freq"],
    noise_sd_units=p["noise_sd_units"]
)

In [5]:
feat, media_medians, non_media_stats = build_features(original, p)

In [7]:
media = p["media_channels"]
non_media = p["non_media_channels"]

In [11]:
np.column_stack([feat[f"{c}_hill"].values for c in media])


array([[0.07976184, 0.18152309, 0.12288035, 0.15417509],
       [0.27330906, 0.32365268, 0.36805979, 0.33188523],
       [0.44528419, 0.41897384, 0.3455411 , 0.43458822],
       [0.55952334, 0.42242035, 0.35694748, 0.46668279],
       [0.60880625, 0.45426768, 0.32793109, 0.4483037 ],
       [0.64757021, 0.46609973, 0.32577437, 0.51593958],
       [0.65986372, 0.47815829, 0.22784472, 0.51776733],
       [0.67871837, 0.5108571 , 0.25456348, 0.51050347],
       [0.70907851, 0.47202381, 0.32182908, 0.53919279],
       [0.735313  , 0.42601483, 0.24310905, 0.53007156],
       [0.73144002, 0.40593739, 0.29441163, 0.55271982],
       [0.73369122, 0.34736852, 0.31105868, 0.55260537],
       [0.73961217, 0.33674527, 0.37971355, 0.54989663],
       [0.74588145, 0.33108833, 0.40519191, 0.49948651],
       [0.74164644, 0.31987391, 0.39526476, 0.5223565 ],
       [0.74710028, 0.32699086, 0.42029746, 0.50821605],
       [0.74342137, 0.30534976, 0.45218652, 0.47954213],
       [0.74304073, 0.34250684,

In [12]:
np.column_stack([feat[f"{c}_scaled"].values for c in non_media])


array([[-4.56271034e-01,  1.52133759e+00, -1.26035754e+00],
       [-5.82533760e-01,  8.00740310e-01, -1.17384602e+00],
       [ 5.00741481e-01,  6.76591911e-01, -1.07021720e+00],
       [-3.33025654e-01,  8.70350345e-01, -9.50982191e-01],
       [-1.18376132e-01, -1.14350946e-01, -8.17879724e-01],
       [ 7.44661224e-01,  3.22849040e-01, -6.72850728e-01],
       [-1.85609301e+00, -1.14812154e+00, -5.18010054e-01],
       [ 1.31276958e+00,  6.03933030e-01, -3.55615627e-01],
       [ 8.98383331e-02, -1.02990982e+00, -1.88035523e-01],
       [-3.85515510e-01,  7.43415839e-01, -1.77134384e-02],
       [-6.91268680e-02, -5.13676657e-01,  1.52866949e-01],
       [-1.84044739e-02, -5.58335345e-01,  3.21218191e-01],
       [-1.68118354e+00, -9.28226232e-01,  4.84885349e-01],
       [ 1.48618784e+00, -8.22905140e-01,  6.41481787e-01],
       [ 5.90546094e-02,  1.37495659e-01,  7.88723976e-01],
       [ 4.53414070e-01, -2.36676894e+00,  9.24464793e-01],
       [ 9.52697882e-01, -1.10380190e-01

In [22]:
X_core = np.column_stack([np.column_stack([feat[f"{c}_hill"].values for c in media])
, np.column_stack([feat[f"{c}_scaled"].values for c in non_media])
])

# intercept column
np.column_stack([np.ones(len(feat)), X_core])

array([[ 1.        ,  0.07976184,  0.18152309, ..., -0.45627103,
         1.52133759, -1.26035754],
       [ 1.        ,  0.27330906,  0.32365268, ..., -0.58253376,
         0.80074031, -1.17384602],
       [ 1.        ,  0.44528419,  0.41897384, ...,  0.50074148,
         0.67659191, -1.0702172 ],
       ...,
       [ 1.        ,  0.6461001 ,  0.38178856, ...,  0.72648076,
         1.63981245, -1.40592734],
       [ 1.        ,  0.67978279,  0.39731841, ..., -0.88110125,
         1.20678811, -1.37725048],
       [ 1.        ,  0.69950444,  0.44613038, ...,  1.3492335 ,
         1.69742597, -1.3284902 ]], shape=(156, 8))